# AeroNet Lite — Anomaly Classifier
### Module 5B · Drone Telemetry Anomaly Detection & Classification

---

This notebook implements the **anomaly detection** component of the AeroNet Lite pipeline.  
It mirrors the logic in `src/ml_pipeline.py → run_anomaly_detection()` and exposes every step visually.

**Pipeline overview:**
1. Generate synthetic drone flight telemetry with four labelled anomaly classes  
2. Exploratory Data Analysis — class balance, feature distributions, pairplot  
3. Train Decision Tree and Random Forest classifiers  
4. Evaluate with accuracy, confusion matrix, and classification report  
5. Visualise the Decision Tree structure  
6. Demonstrate the live rule-based classifier used inside the simulation  

**Anomaly classes:**

| Label | Class | Key signal |
|-------|-------|------------|
| 0 | Normal | All features in baseline range |
| 1 | Battery anomaly | `battery_drop` > 15 % (sudden drain) |
| 2 | Route anomaly | `route_deviation` > 4 cells |
| 3 | Sensor spike | `altitude_change` > 8 m  or  `speed_change` > 12 m/s |

---
## 0 · Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family':      'monospace',
})
CLASS_COLORS = ['#3fb950', '#f78166', '#58a6ff', '#d2a8ff']
CLASS_NAMES  = ['Normal', 'Battery anomaly', 'Route anomaly', 'Sensor spike']
RANDOM_STATE = 42

print('Environment ready ✓')

---
## 1 · Telemetry Data Generation

The synthetic dataset replicates the generator in `ml_pipeline.py → generate_telemetry_data()`.  
Each class occupies a distinct region of feature space — this is intentional to demonstrate the pipeline.  
Real UAV telemetry would show overlapping distributions and lower accuracy.

In [ ]:
def generate_telemetry_data(n: int = 1000, seed: int = 42) -> pd.DataFrame:
    """
    Synthetic drone flight telemetry with four labelled anomaly classes.

    Features
    --------
    battery_drop      : % battery lost in last 10 s
    speed             : ground speed in m/s
    route_deviation   : lateral deviation from planned path in cells
    altitude_change   : vertical displacement in last tick (m)
    speed_change      : speed delta in last tick (m/s)

    Labels
    ------
    0 = Normal          — all values in baseline range
    1 = Battery anomaly — high battery_drop (sudden drain)
    2 = Route anomaly   — high route_deviation (off-course)
    3 = Sensor spike    — extreme altitude_change or speed_change
    """
    rng = np.random.default_rng(seed)
    n_per_class = n // 4
    rows = []

    for label in range(4):
        for _ in range(n_per_class):
            if label == 0:      # Normal
                bd  = rng.uniform(1, 5)
                spd = rng.uniform(8, 14)
                dev = rng.uniform(0, 1)
                alt = rng.uniform(-0.5, 0.5)
                sc  = rng.uniform(-1, 1)
            elif label == 1:    # Battery anomaly
                bd  = rng.uniform(18, 40)   # sudden high drain
                spd = rng.uniform(6, 12)
                dev = rng.uniform(0, 1.5)
                alt = rng.uniform(-1, 1)
                sc  = rng.uniform(-1, 1)
            elif label == 2:    # Route anomaly
                bd  = rng.uniform(1, 6)
                spd = rng.uniform(8, 14)
                dev = rng.uniform(5, 15)    # large lateral deviation
                alt = rng.uniform(-1, 1)
                sc  = rng.uniform(-1, 1)
            else:               # Sensor spike
                bd  = rng.uniform(1, 6)
                spd = rng.uniform(8, 14)
                dev = rng.uniform(0, 2)
                alt = rng.uniform(10, 40)   # altitude spike
                sc  = rng.uniform(15, 30)   # speed spike
            rows.append([bd, spd, dev, alt, sc, label])

    df = pd.DataFrame(
        rows,
        columns=['battery_drop', 'speed', 'route_deviation',
                 'altitude_change', 'speed_change', 'label']
    )
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


df = generate_telemetry_data(n=1000, seed=RANDOM_STATE)
print(f'Dataset shape : {df.shape}')
print(f'Class balance :')
print(df['label'].value_counts().sort_index().rename(
    {i: CLASS_NAMES[i] for i in range(4)}
))
df.head(10)

---
## 2 · Exploratory Data Analysis

In [ ]:
print('=== Descriptive Statistics by Class ===')
df['class_name'] = df['label'].map({i: n for i, n in enumerate(CLASS_NAMES)})
display(df.groupby('class_name').describe().round(2))
df.drop(columns='class_name', inplace=True)

In [ ]:
FEATURES = ['battery_drop', 'speed', 'route_deviation', 'altitude_change', 'speed_change']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Feature Distributions by Anomaly Class', fontsize=13, fontweight='bold')

for ax, feat in zip(axes, FEATURES):
    for label, color, name in zip(range(4), CLASS_COLORS, CLASS_NAMES):
        subset = df[df['label'] == label][feat]
        ax.hist(subset, bins=25, alpha=0.65, color=color, label=name, edgecolor='none')
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('Value')
    ax.grid(True, alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('Count')

handles = [mpatches.Patch(color=c, label=n)
           for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=handles, loc='upper right', ncol=4,
           bbox_to_anchor=(1.0, 1.02), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Feature Boxplots by Anomaly Class', fontsize=13, fontweight='bold')

for ax, feat in zip(axes, FEATURES):
    data_by_class = [df[df['label'] == i][feat].values for i in range(4)]
    bp = ax.boxplot(data_by_class, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], CLASS_COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticklabels([n[:7] for n in CLASS_NAMES], rotation=15, fontsize=8)
    ax.set_title(feat, fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# 2-D scatter of the two most discriminating features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Key Feature Separability', fontsize=13, fontweight='bold')

for ax, x_feat, y_feat in [
    (axes[0], 'battery_drop',    'route_deviation'),
    (axes[1], 'altitude_change', 'speed_change'),
]:
    for label, color, name in zip(range(4), CLASS_COLORS, CLASS_NAMES):
        sub = df[df['label'] == label]
        ax.scatter(sub[x_feat], sub[y_feat],
                   alpha=0.45, s=18, color=color, label=name)
    ax.set_xlabel(x_feat)
    ax.set_ylabel(y_feat)
    ax.set_title(f'{x_feat}  vs  {y_feat}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3 · Train / Test Split

In [ ]:
X = df[FEATURES].values
y = df['label'].values

# 75/25 split — mirrors ml_pipeline.py
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples     : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print()
print('Class distribution (train):')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASS_NAMES[u]:<20} {c} samples')

---
## 4 · Model Training

Two classifiers, matching `run_anomaly_detection()` in `ml_pipeline.py`:
- **Decision Tree** — `max_depth=6`, human-interpretable splits  
- **Random Forest** — 100 trees, robust ensemble

In [ ]:
# ── Decision Tree ─────────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc  = accuracy_score(y_test, dt_pred)

print(f'Decision Tree (max_depth=6)')
print(f'  Accuracy  : {dt_acc*100:.2f}%')
print(f'  Depth used: {dt.get_depth()}')
print(f'  Leaves    : {dt.get_n_leaves()}')

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)

print(f'Random Forest (100 trees)')
print(f'  Accuracy : {rf_acc*100:.2f}%')

best_pred  = rf_pred if rf_acc >= dt_acc else dt_pred
best_label = 'Random Forest' if rf_acc >= dt_acc else 'Decision Tree'
best_acc   = max(rf_acc, dt_acc)
print(f'\n→ Best model: {best_label}  (Accuracy = {best_acc*100:.2f}%)')

---
## 5 · Evaluation

In [ ]:
results = pd.DataFrame({
    'Model':    ['Decision Tree', 'Random Forest'],
    'Accuracy': [dt_acc * 100,    rf_acc * 100],
})
print('=== Model Comparison ===')
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(results['Model'], results['Accuracy'],
              color=['#58a6ff', '#f78166'], alpha=0.85, width=0.4)
for bar, val in zip(bars, results['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Classifier Accuracy Comparison', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')

for ax, preds, title in [
    (axes[0], dt_pred, f'Decision Tree  (acc={dt_acc*100:.2f}%)'),
    (axes[1], rf_pred, f'Random Forest  (acc={rf_acc*100:.2f}%)'),
]:
    cm = confusion_matrix(y_test, preds)
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    tick_marks = np.arange(len(CLASS_NAMES))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    short_names = ['Normal', 'Battery', 'Route', 'Sensor']
    ax.set_xticklabels(short_names, rotation=20, ha='right', fontsize=9)
    ax.set_yticklabels(short_names, fontsize=9)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else '#e6edf3',
                    fontsize=13, fontweight='bold')
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
print(f'=== Classification Report — {best_label} ===')
print(classification_report(y_test, best_pred,
                             target_names=CLASS_NAMES,
                             zero_division=0))

---
## 6 · Feature Importances

In [ ]:
importances = rf.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ['#3fb950', '#58a6ff', '#ffa657', '#f78166', '#d2a8ff']
bars = ax.barh(feat_df['feature'], feat_df['importance'],
               color=bar_colors, alpha=0.85, height=0.55)
for bar, val in zip(bars, feat_df['importance']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Feature ranking:')
print(feat_df.sort_values('importance', ascending=False).to_string(index=False))

---
## 7 · Decision Tree Visualisation

The trained Decision Tree's first 3 levels reveal how the model splits on the key discriminating features.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

plot_tree(
    dt,
    max_depth=3,
    feature_names=FEATURES,
    class_names=['Normal', 'Battery', 'Route', 'Sensor'],
    filled=True,
    rounded=True,
    impurity=True,
    ax=ax,
    fontsize=8,
)
ax.set_title('Decision Tree — First 3 Levels (max_depth=6 total)',
             fontsize=13, fontweight='bold', color='#e6edf3', pad=12)
plt.tight_layout()
plt.show()

print('Root split feature:', FEATURES[dt.tree_.feature[0]])
print(f'Root split threshold: {dt.tree_.threshold[0]:.4f}')

---
## 8 · Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_dt = cross_val_score(
    DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    X, y, cv=skf, scoring='accuracy'
)
cv_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    X, y, cv=skf, scoring='accuracy'
)

print('5-Fold Stratified Cross-Validation Accuracy')
print(f'  Decision Tree : {cv_dt.mean()*100:.2f}% ± {cv_dt.std()*100:.2f}%')
print(f'  Random Forest : {cv_rf.mean()*100:.2f}% ± {cv_rf.std()*100:.2f}%')

fig, ax = plt.subplots(figsize=(9, 4))
bp = ax.boxplot([cv_dt * 100, cv_rf * 100], positions=[1, 2],
                patch_artist=True, widths=0.4,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], ['#58a6ff', '#f78166']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Decision Tree', 'Random Forest'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('5-Fold CV Accuracy Distribution', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 9 · Live Rule-Based Classifier

During the simulation (`delivery_simulator.py`) a lightweight rule-based classifier — `classify_drone_reading()` — is used for zero-latency anomaly tagging.  
The rules mirror the synthetic data generation boundaries exactly.

In [ ]:
def classify_drone_reading(
    battery_drop: float,
    speed: float,
    route_deviation: float,
    altitude_change: float,
    speed_change: float
) -> str:
    """
    Rule-based anomaly classifier — mirrors ml_pipeline.classify_drone_reading().
    Used for live in-simulation classification (no model inference overhead).
    
    Priority order:
      1. Battery anomaly  — battery_drop > 15
      2. Route anomaly    — route_deviation > 4
      3. Sensor spike     — |altitude_change| > 8  or  |speed_change| > 12
      4. Normal
    """
    if battery_drop > 15:
        return 'Battery anomaly'
    if route_deviation > 4:
        return 'Route anomaly'
    if abs(altitude_change) > 8 or abs(speed_change) > 12:
        return 'Sensor spike'
    return 'Normal'


# ── Test cases from each class ────────────────────────────────────────────────
test_cases = [
    # (battery_drop, speed, route_deviation, altitude_change, speed_change, expected)
    (2.1,  11.5,  0.3,   0.1,  0.5,  'Normal'),
    (25.0, 9.0,   0.5,   0.2,  0.3,  'Battery anomaly'),
    (3.0,  12.0,  8.5,   0.0,  0.0,  'Route anomaly'),
    (1.5,  11.0,  1.2,  15.0, 18.0,  'Sensor spike'),
    (4.9,  13.1,  0.8,  -0.3,  0.9,  'Normal'),
    (19.5,  8.5,  6.0,  12.0, 20.0,  'Battery anomaly'),  # battery takes priority
]

print(f'{"battery_drop":>13} {"route_dev":>10} {"alt_chg":>8} {"spd_chg":>8}   {"Expected":<20} {"Got":<20} {"✓/✗"}')
print('-' * 90)
all_correct = True
for bd, spd, dev, alt, sc, expected in test_cases:
    got = classify_drone_reading(bd, spd, dev, alt, sc)
    ok  = '✓' if got == expected else '✗'
    if got != expected:
        all_correct = False
    print(f'{bd:>13.1f} {dev:>10.1f} {alt:>8.1f} {sc:>8.1f}   {expected:<20} {got:<20} {ok}')
print()
print('All test cases passed ✓' if all_correct else 'Some test cases FAILED ✗')

In [ ]:
# Validate rule-based classifier against the full test set
rule_preds = [
    classify_drone_reading(row[0], row[1], row[2], row[3], row[4])
    for row in X_test
]
rule_labels = [CLASS_NAMES.index(p) for p in rule_preds]
rule_acc = accuracy_score(y_test, rule_labels)

print('Rule-based classifier accuracy on test set:', f'{rule_acc*100:.2f}%')
print()
print('Agreement between Rule-based and Random Forest:')
agreement = np.mean(np.array(rule_labels) == rf_pred)
print(f'  {agreement*100:.2f}% of predictions match')

---
## 10 · Simulation Integration Demo

Simulate 10 drone telemetry readings (as the delivery simulator would inject at Step 18).

In [ ]:
rng = np.random.default_rng(99)

sim_readings = []
for i in range(10):
    # Mix of normal and anomalous readings
    bd  = float(rng.choice([rng.uniform(1, 5), rng.uniform(18, 35)], p=[0.7, 0.3]))
    spd = float(rng.uniform(7, 15))
    dev = float(rng.choice([rng.uniform(0, 1), rng.uniform(5, 12)], p=[0.75, 0.25]))
    alt = float(rng.choice([rng.uniform(-0.5, 0.5), rng.uniform(10, 35)], p=[0.8, 0.2]))
    sc  = float(rng.choice([rng.uniform(-1, 1), rng.uniform(15, 28)], p=[0.8, 0.2]))

    rule_label  = classify_drone_reading(bd, spd, dev, alt, sc)
    model_label = CLASS_NAMES[rf.predict([[bd, spd, dev, alt, sc]])[0]]

    sim_readings.append({
        'Drone':        f'Drone-{i+1:02d}',
        'battery_drop': round(bd, 2),
        'speed':        round(spd, 2),
        'route_dev':    round(dev, 2),
        'alt_change':   round(alt, 2),
        'spd_change':   round(sc, 2),
        'Rule':         rule_label,
        'RF Model':     model_label,
        'Match':        '✓' if rule_label == model_label else '✗',
    })

sim_df = pd.DataFrame(sim_readings)
print('=== Simulated Telemetry Readings ===')
print(sim_df.to_string(index=False))
print()
anomalies = sim_df[sim_df['Rule'] != 'Normal']
print(f'Anomalies detected: {len(anomalies)} / {len(sim_df)}')
print(anomalies[['Drone', 'Rule']].to_string(index=False))

---
## 11 · Summary

In [ ]:
print('=' * 60)
print('     ANOMALY CLASSIFIER — FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset rows       : {len(df):,}  (250 per class)')
print(f'  Train/Test split   : 75% / 25%')
print(f'  Features used      : {FEATURES}')
print(f'  Classes            : {CLASS_NAMES}')
print()
print(f'  {"Model":<22} {"Accuracy":>10}')
print(f'  {"-"*35}')
print(f'  {"Decision Tree":<22} {dt_acc*100:>9.2f}%')
print(f'  {"Random Forest":<22} {rf_acc*100:>9.2f}%')
print(f'  {"Rule-based (live)":<22} {rule_acc*100:>9.2f}%')
print()
print(f'  Best model  : {best_label}  (Accuracy = {best_acc*100:.2f}%)')
print(f'  5-CV Accuracy: DT = {cv_dt.mean()*100:.2f}% ± {cv_dt.std()*100:.2f}%')
print(f'                 RF = {cv_rf.mean()*100:.2f}% ± {cv_rf.std()*100:.2f}%')
print()
print('Key observations:')
print('  • battery_drop and route_deviation are the most important features')
print('  • Synthetic data has non-overlapping class boundaries → near 100% accuracy')
print('  • Rule-based classifier agrees with RF model on most samples')
print('  • Real UAV telemetry would yield lower accuracy due to sensor noise')
print('  • Anomaly detection fires at Step 18 of the AeroNet Lite simulation')
print('=' * 60)